# 29 · 手撸最小 MCP 协议服务器

> **学习目标**：理解 MCP（Model Context Protocol）本质是什么 —— **JSON-RPC 2.0 + 几个标准方法**。手撸一个 in-process MCP-style server，让 Agent 通过协议调工具，而不是直接 import 函数。
>
> **预备**：26、28 跑过。本机 `mcp` Python SDK 没装（网络不通装不上），所以**手撸**。
>
> **为什么重要**：MCP 是 Anthropic 主推的「Agent 工具标准化协议」，让同一个 LLM 客户端能接任何人写的工具服务器。理解协议本质 → 你能 5 分钟写个 server，也能看懂别人写的 client。

**MCP 一句话**：tools / resources / prompts 三类东西的标准化暴露协议，**transport** 可以是 stdio（subprocess）/ HTTP / WebSocket。

In [1]:
import json, uuid
from typing import Callable, Any

## 1. JSON-RPC 2.0 —— MCP 的传输层语法

**核心：3 种消息**：

| 类型 | 字段 | 说明 |
|------|------|------|
| Request | `id`, `method`, `params` | 客户端发，要求响应 |
| Response | `id`, `result` 或 `error` | 服务端回应（id 与 Request 匹配）|
| Notification | `method`, `params`（**无 id**）| 单向通知，不要求响应 |

**MCP 在 JSON-RPC 之上规定的标准方法**：
- `initialize` —— handshake，交换 protocol_version / capabilities
- `tools/list` —— 列出可用工具
- `tools/call` —— 调用某个工具
- `resources/list` / `resources/read` —— 暴露资源（文件 / DB / API 端点）
- `prompts/list` / `prompts/get` —— 暴露 prompt 模板

In [2]:
# JSON-RPC 消息构造
def make_request(method: str, params: dict | None = None, req_id: str | None = None) -> dict:
    return {
        'jsonrpc': '2.0',
        'id': req_id or uuid.uuid4().hex[:8],
        'method': method,
        'params': params or {},
    }

def make_response(req_id: str, result: Any = None, error: dict | None = None) -> dict:
    msg = {'jsonrpc': '2.0', 'id': req_id}
    if error: msg['error'] = error
    else:     msg['result'] = result
    return msg

def make_notification(method: str, params: dict | None = None) -> dict:
    return {'jsonrpc': '2.0', 'method': method, 'params': params or {}}

# 演示
print('Request   :', json.dumps(make_request('tools/list'), ensure_ascii=False))
print('Response  :', json.dumps(make_response('abc123', {'tools': []}), ensure_ascii=False))
print('Notification:', json.dumps(make_notification('progress', {'percent': 50}), ensure_ascii=False))

Request   : {"jsonrpc": "2.0", "id": "d0a96c23", "method": "tools/list", "params": {}}
Response  : {"jsonrpc": "2.0", "id": "abc123", "result": {"tools": []}}
Notification: {"jsonrpc": "2.0", "method": "progress", "params": {"percent": 50}}


## 2. 手撸 MCP server —— 50 行内

**关键设计**：dispatch table + 标准错误码。

In [3]:
# JSON-RPC 标准错误码（部分）
ERR_PARSE         = -32700
ERR_INVALID_REQ   = -32600
ERR_METHOD_NOT_FOUND = -32601
ERR_INVALID_PARAMS   = -32602
ERR_INTERNAL      = -32603

class MCPServer:
    """最小 MCP-like server。in-process，generic 到任何 transport。"""
    PROTOCOL_VERSION = '2024-11-05'

    def __init__(self, name: str, version: str = '0.1.0'):
        self.name = name
        self.version = version
        self.tools: dict[str, dict] = {}   # name -> {desc, input_schema, fn}
        self.resources: dict[str, dict] = {}   # uri -> {name, mime, fn}

    # ---- 注册（server 启动时调） ----
    def add_tool(self, name: str, description: str, input_schema: dict, fn: Callable):
        self.tools[name] = {'description': description, 'input_schema': input_schema, 'fn': fn}

    def add_resource(self, uri: str, name: str, mime: str, fn: Callable):
        self.resources[uri] = {'name': name, 'mime_type': mime, 'fn': fn}

    # ---- 协议方法 ----
    def handle(self, msg: dict) -> dict | None:
        """处理一条 JSON-RPC 消息，返回响应（notification 返回 None）。"""
        req_id = msg.get('id')
        method = msg.get('method')
        params = msg.get('params') or {}

        if method == 'initialize':
            return make_response(req_id, {
                'protocolVersion': self.PROTOCOL_VERSION,
                'serverInfo': {'name': self.name, 'version': self.version},
                'capabilities': {'tools': {}, 'resources': {}},
            })
        if method == 'tools/list':
            return make_response(req_id, {
                'tools': [
                    {'name': n, 'description': t['description'], 'inputSchema': t['input_schema']}
                    for n, t in self.tools.items()
                ]
            })
        if method == 'tools/call':
            name = params.get('name'); args = params.get('arguments') or {}
            if name not in self.tools:
                return make_response(req_id, error={'code': ERR_METHOD_NOT_FOUND, 'message': f'unknown tool: {name}'})
            try:
                content = self.tools[name]['fn'](**args)
                return make_response(req_id, {'content': [{'type': 'text', 'text': str(content)}], 'isError': False})
            except TypeError as e:
                return make_response(req_id, error={'code': ERR_INVALID_PARAMS, 'message': str(e)})
            except Exception as e:
                return make_response(req_id, {'content': [{'type': 'text', 'text': str(e)}], 'isError': True})
        if method == 'resources/list':
            return make_response(req_id, {
                'resources': [
                    {'uri': uri, 'name': r['name'], 'mimeType': r['mime_type']}
                    for uri, r in self.resources.items()
                ]
            })
        if method == 'resources/read':
            uri = params.get('uri')
            if uri not in self.resources:
                return make_response(req_id, error={'code': ERR_INVALID_PARAMS, 'message': f'unknown uri: {uri}'})
            content = self.resources[uri]['fn']()
            return make_response(req_id, {
                'contents': [{'uri': uri, 'mimeType': self.resources[uri]['mime_type'], 'text': str(content)}]
            })
        # 未知方法
        return make_response(req_id, error={'code': ERR_METHOD_NOT_FOUND, 'message': f'unknown method: {method}'})

## 3. 注册工具 + 跑通 4 个端到端调用

建一个「文件操作 MCP server」，含 2 个工具 + 1 个资源。

In [4]:
import math, time

# 业务函数
def calc(expression: str) -> str:
    import re
    if not re.fullmatch(r'[\d\s+\-*/().%]+', expression):
        raise ValueError('expression contains forbidden chars')
    return str(eval(expression))

def now() -> str:
    return time.strftime('%Y-%m-%d %H:%M:%S')

def server_info_text() -> str:
    return 'This is a demo MCP server.\nCapabilities: calc, now.\nVersion 0.1.0.'

# 启动 server
server = MCPServer(name='demo-mcp-server')
server.add_tool('calc',
                description='Evaluate a math expression',
                input_schema={
                    'type': 'object',
                    'properties': {'expression': {'type': 'string'}},
                    'required': ['expression'],
                },
                fn=calc)
server.add_tool('now',
                description='Get current time string',
                input_schema={'type': 'object', 'properties': {}},
                fn=now)
server.add_resource('info://about', name='Server Info', mime='text/plain', fn=server_info_text)

print(f'MCP server "{server.name}" 已就绪：{len(server.tools)} tools + {len(server.resources)} resources')

MCP server "demo-mcp-server" 已就绪：2 tools + 1 resources


In [5]:
# 模拟客户端发 4 个请求
def send(method, params=None):
    req = make_request(method, params)
    print(f'→ {method:18}  {json.dumps(params or {}, ensure_ascii=False)}')
    resp = server.handle(req)
    pretty = json.dumps(resp.get('result') or resp.get('error'), ensure_ascii=False)
    print(f'← {pretty[:200]}')
    print()
    return resp

# 1. handshake
send('initialize', {'protocolVersion': '2024-11-05', 'clientInfo': {'name': 'demo-client'}})
# 2. list tools
send('tools/list')
# 3. call tool
send('tools/call', {'name': 'calc', 'arguments': {'expression': '2 * (3 + 4)'}})
send('tools/call', {'name': 'now'})
# 4. error 路径
send('tools/call', {'name': 'unknown_tool'})
send('tools/call', {'name': 'calc', 'arguments': {'expression': 'os.system("hi")'}})
# 5. resources
send('resources/list')
send('resources/read', {'uri': 'info://about'})

→ initialize          {"protocolVersion": "2024-11-05", "clientInfo": {"name": "demo-client"}}
← {"protocolVersion": "2024-11-05", "serverInfo": {"name": "demo-mcp-server", "version": "0.1.0"}, "capabilities": {"tools": {}, "resources": {}}}

→ tools/list          {}
← {"tools": [{"name": "calc", "description": "Evaluate a math expression", "inputSchema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}, {"name": "now

→ tools/call          {"name": "calc", "arguments": {"expression": "2 * (3 + 4)"}}
← {"content": [{"type": "text", "text": "14"}], "isError": false}

→ tools/call          {"name": "now"}
← {"content": [{"type": "text", "text": "2026-06-04 17:59:31"}], "isError": false}

→ tools/call          {"name": "unknown_tool"}
← {"code": -32601, "message": "unknown tool: unknown_tool"}

→ tools/call          {"name": "calc", "arguments": {"expression": "os.system(\"hi\")"}}
← {"content": [{"type": "text", "text": "expression contain

{'jsonrpc': '2.0',
 'id': 'e7eb995b',
 'result': {'contents': [{'uri': 'info://about',
    'mimeType': 'text/plain',
    'text': 'This is a demo MCP server.\nCapabilities: calc, now.\nVersion 0.1.0.'}]}}

## 4. 客户端封装 —— 把 MCP server 当成 Agent 工具源

**关键思路**：Agent 不直接 import 工具函数，而是通过 MCP 协议「**发现 + 调用**」。**好处**：换 server 实现 / 加 server 都不动 Agent。

In [6]:
class MCPClient:
    def __init__(self, server: MCPServer):
        # in-process transport：直接持有 server 引用。
        # 真生产里这里是 subprocess + stdio 或 HTTP。
        self.server = server
        self._initialized = False
        self._tools_cache = None

    def _call(self, method, params=None) -> dict:
        return self.server.handle(make_request(method, params))

    def initialize(self):
        r = self._call('initialize', {'protocolVersion': MCPServer.PROTOCOL_VERSION})
        self._initialized = True
        return r['result']

    def list_tools(self) -> list[dict]:
        if self._tools_cache is None:
            r = self._call('tools/list')
            self._tools_cache = r['result']['tools']
        return self._tools_cache

    def call_tool(self, name: str, **arguments) -> str:
        r = self._call('tools/call', {'name': name, 'arguments': arguments})
        if 'error' in r:
            raise RuntimeError(f'MCP error: {r["error"]}')
        content = r['result']['content']
        return content[0]['text'] if content else ''

# 用一下
client = MCPClient(server)
info = client.initialize()
print(f'已连接 server: {info["serverInfo"]}')
tools = client.list_tools()
print(f'\nServer 暴露 {len(tools)} 工具：')
for t in tools:
    print(f'  - {t["name"]}: {t["description"]}')

print(f'\n调用 calc("100*1.05"):  {client.call_tool("calc", expression="100*1.05")}')
print(f'调用 now():            {client.call_tool("now")}')

已连接 server: {'name': 'demo-mcp-server', 'version': '0.1.0'}

Server 暴露 2 工具：
  - calc: Evaluate a math expression
  - now: Get current time string

调用 calc("100*1.05"):  105.0
调用 now():            2026-06-04 17:59:31


In [7]:
# 把 MCP client 的工具列表自动转成 OpenAI tools 格式（26 号 notebook 风格）
def mcp_tools_to_openai(client: MCPClient) -> list[dict]:
    return [
        {
            'type': 'function',
            'function': {
                'name': t['name'],
                'description': t['description'],
                'parameters': t['inputSchema'],
            }
        }
        for t in client.list_tools()
    ]

openai_tools = mcp_tools_to_openai(client)
print('转换后的 OpenAI 风格 tools：')
print(json.dumps(openai_tools, ensure_ascii=False, indent=2))
print('\n→ 这就是「接 MCP 接哪都一样」的本质：MCP server 给标准 schema，Agent 直接用。')

转换后的 OpenAI 风格 tools：
[
  {
    "type": "function",
    "function": {
      "name": "calc",
      "description": "Evaluate a math expression",
      "parameters": {
        "type": "object",
        "properties": {
          "expression": {
            "type": "string"
          }
        },
        "required": [
          "expression"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "now",
      "description": "Get current time string",
      "parameters": {
        "type": "object",
        "properties": {}
      }
    }
  }
]

→ 这就是「接 MCP 接哪都一样」的本质：MCP server 给标准 schema，Agent 直接用。


## 5. Transport 层 —— 从 in-process 到真 MCP

我们的 demo 是「in-process」（client 直接持有 server 实例）。**真 MCP server 跑在独立进程**：

### 5.1 stdio transport（**最常见**，Claude Code 用这套）
Client subprocess.Popen 启动 server 进程，**通过 stdin / stdout 收发 JSON-RPC 消息**（每行一条 JSON）。

```python
# 伪代码
import subprocess, json
p = subprocess.Popen(['python', 'my_mcp_server.py'],
                     stdin=subprocess.PIPE, stdout=subprocess.PIPE)
p.stdin.write(json.dumps(make_request('initialize')).encode() + b'\n')
p.stdin.flush()
response_line = p.stdout.readline()
response = json.loads(response_line)
```

### 5.2 HTTP transport
Server 跑 HTTP service（POST JSON-RPC body 到某 endpoint）。**便于远程**、容易调试，**Anthropic 推 Streamable HTTP**。

### 5.3 我们的 demo 怎么改成 stdio？
```python
import sys
server = MCPServer('my-server')
# ...add_tool...
for line in sys.stdin:
    msg = json.loads(line)
    resp = server.handle(msg)
    if resp:
        print(json.dumps(resp), flush=True)
```
**就这 6 行**就能把本 notebook 的 `MCPServer` 类变成一个真 MCP server，Claude Code 一配就接上。

## 深入思考

1. **MCP vs LangChain Tools 区别？**
   - LangChain Tools 是 **Python 类**，必须 import；MCP 是 **跨进程协议**，server 可以用任何语言写。MCP **生态意义** = 像 USB 一样跨厂商。
2. **为什么有 `initialize` 这一步？**
   - 交换 protocol_version（防止版本不兼容）+ capabilities（client 知道 server 支持哪些可选特性）。**像 TCP 的 3 次握手**。
3. **MCP 的 resources 和 tools 怎么分？**
   - tools = 可执行的动作（有副作用 / 有计算）；resources = 可读取的数据（文件、API 端点、DB 记录）。生产里同一个能力既可以包成 tool 也可以包成 resource，**取决于「谁主动取」**。
4. **MCP server 能 stateful 吗？**
   - 能。`session_id` 透传 + server 自管状态。但**默认推荐 stateless** —— 易扩展、易调试。
5. **Anthropic 为什么主推 MCP 而不是统一一个 SDK？**
   - SDK = 锁在 Anthropic；协议 = 任何 LLM 客户端能接，**生态规模化**。这是「平台」与「服务」的差别。

**改一改**：
- 在 server 加第 3 个工具 `read_file(path)`，让 client list_tools 后自动看到
- 修改 `MCPServer.handle` 加 access log（每次调用打 method / params / latency），观察行为

## 自检 ✅

- [ ] 默写 JSON-RPC Request / Response / Notification 三种消息格式
- [ ] 列出 MCP 必备的 3 个 method（initialize / tools/list / tools/call）
- [ ] 解释 MCP tools vs resources 的区别
- [ ] 解释「为什么 Anthropic 推 MCP 而不是另一个 SDK」
- [ ] 6 行内把 in-process server 改成 stdio MCP server（默背）

## 下一步

进入 Stage 3 → [`../stage3_高级/30_manager_worker_multiagent.ipynb`](../stage3_高级/30_manager_worker_multiagent.ipynb)